In [ ]:
import sqlite3
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Đường dẫn file CSV đầu vào và CSDL đích
WORK_DIR = Path(r"E:\tailieu\HQTCSDL\1_BA\6_nopbai\Tuan_02_04_Xu_ly_du_lieu_UAV")
if not WORK_DIR.exists():
    WORK_DIR = Path.cwd() if Path.cwd().name.startswith("Tuan_") else Path.cwd() / "Tuan_02_04_Xu_ly_du_lieu_UAV"
CSV_PATH = WORK_DIR / "20181029_dX_0830_0900.csv"
DB_PATH = Path(r"E:\tailieu\HQTCSDL\1_BA\6_nopbai\database\UrbanTrafficAnalyticsDB.db")
if not DB_PATH.parent.exists():
    DB_PATH = (Path.cwd().parent if Path.cwd().name.startswith("Tuan_") else Path.cwd()) / "database" / "UrbanTrafficAnalyticsDB.db"

# 1. Kiểm tra thông tin file gốc và đọc dòng tiêu đề + dòng đầu tiên
file_size_mb = CSV_PATH.stat().st_size / (1024 ** 2)
with open(CSV_PATH, "r", encoding="utf-8", errors="replace") as f:
    header_line = f.readline().strip()
    first_vehicle_line = f.readline().strip()

first_parts = [p.strip() for p in first_vehicle_line.rstrip("; ").split(";")]
print(f"📄 File đầu vào : {CSV_PATH.name} (Dung lượng: {file_size_mb:,.1f} MB)")
print(f"📌 Dòng tiêu đề : {header_line}")
print(f"🚗 Ví dụ Dòng 1 : track_id={first_parts[0]}, type={first_parts[1]}, traveled_d={first_parts[2]}m, avg_speed={first_parts[3]}km/h")
print(f"   • Tổng số giá trị trên Dòng 1: {len(first_parts):,} cột -> Tương ứng {(len(first_parts)-4)//6:,} điểm quỹ đạo (25 Hz)")
print(f"   • Điểm đầu tiên (t=0.00s): lat={first_parts[4]}, lon={first_parts[5]}, speed={first_parts[6]} km/h, lon_acc={first_parts[7]}, lat_acc={first_parts[8]}, time={first_parts[9]}s")

In [ ]:
t0 = time.perf_counter()
STEP_FRAMES = 25  # Tần số quay gốc 25 Hz -> lấy mỗi 25 khung hình = 1 điểm / 1 giây (1 Hz)

raw_points = []
malformed_cells = 0

with open(CSV_PATH, "r", encoding="utf-8", errors="replace") as f:
    _ = f.readline()  # Bỏ qua dòng tiêu đề
    for line in f:
        parts = line.rstrip("\r\n; ").split(";")
        if len(parts) < 10:
            continue
        try:
            tid = int(float(parts[0].strip()))
            vtype = parts[1].strip()
            traveled_d = float(parts[2].strip())
            avg_spd = float(parts[3].strip())
        except ValueError:
            continue

        # Số điểm quỹ đạo 25Hz của phương tiện hiện tại (mỗi điểm gồm 6 cột)
        n_pts = (len(parts) - 4) // 6

        # Lấy mẫu cách đều mỗi STEP_FRAMES (1 giây / 1 điểm)
        for idx in range(0, n_pts, STEP_FRAMES):
            b = 4 + idx * 6
            try:
                raw_points.append((
                    tid,
                    vtype,
                    traveled_d,
                    avg_spd,
                    float(parts[b]),       # lat
                    float(parts[b + 1]),   # lon
                    float(parts[b + 2]),   # speed (km/h)
                    float(parts[b + 3]),   # lon_acc (m/s^2)
                    float(parts[b + 4]),   # lat_acc (m/s^2)
                    float(parts[b + 5]),   # time (seconds)
                ))
            except ValueError:
                malformed_cells += 1
                raw_points.append((tid, vtype, traveled_d, avg_spd, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan))

df_raw = pd.DataFrame(
    raw_points,
    columns=[
        "source_track_id", "vehicle_type", "traveled_d_m", "avg_speed_kmh",
        "lat", "lon", "speed_kmh", "lon_acc_ms2", "lat_acc_ms2", "time_sec"
    ]
)

elapsed_read = time.perf_counter() - t0
print(f"✅ Đã đọc và chuyển đổi xong trong {elapsed_read:.2f} giây!")
print(f"   • Tổng số phương tiện (Tracks) ban đầu : {df_raw['source_track_id'].nunique():,} xe")
print(f"   • Tổng số điểm quỹ đạo (1 Hz) ban đầu  : {len(df_raw):,} điểm")
print(f"   • Khoảng thời gian ghi nhận            : {df_raw['time_sec'].min():.1f}s -> {df_raw['time_sec'].max():.2f}s (~{df_raw['time_sec'].max()/60:.1f} phút)")

print("\n🔍 Xem 5 dòng đầu của bảng dữ liệu thô sau khi chuyển sang dạng chuẩn (Long Format):")
display(df_raw.head(5))

print("\n📊 Thống kê số lượng xe và số điểm quỹ đạo theo từng loại phương tiện ban đầu:")
veh_init_summary = (
    df_raw.groupby("vehicle_type", as_index=False)
    .agg(
        So_Phuong_Tien=("source_track_id", "nunique"),
        So_Diem_Quy_Dao_1Hz=("time_sec", "count"),
        Quang_Duong_TB_m=("traveled_d_m", "mean"),
        Van_Toc_TB_kmh=("speed_kmh", "mean"),
    )
    .sort_values("So_Phuong_Tien", ascending=False)
    .round(2)
)
display(veh_init_summary)

In [ ]:
# Lưu số lượng ban đầu để lập bảng đối chiếu trước/sau làm sạch
n0_pts, n0_veh = len(df_raw), df_raw["source_track_id"].nunique()

# Bước 1: Loại bỏ giá trị thiếu (NaN) và bản ghi trùng lặp (cùng xe, cùng thời điểm)
df_step1 = df_raw.dropna(subset=["source_track_id", "lat", "lon", "speed_kmh", "time_sec"]).copy()
df_step1 = df_step1.drop_duplicates(subset=["source_track_id", "time_sec"], keep="first")
n1_pts, n1_veh = len(df_step1), df_step1["source_track_id"].nunique()

# Bước 2: Loại bỏ xe đỗ tĩnh ven đường (tổng quãng đường di chuyển traveled_d_m < 5.0 mét)
df_step2 = df_step1[df_step1["traveled_d_m"] >= 5.0].copy()
n2_pts, n2_veh = len(df_step2), df_step2["source_track_id"].nunique()

# Bước 3: Lọc bỏ điểm nhiễu vận tốc (0 - 80 km/h) và gia tốc (|acc| <= 8 m/s^2)
cond_speed = (df_step2["speed_kmh"] >= 0.0) & (df_step2["speed_kmh"] <= 80.0)
cond_acc = (df_step2["lon_acc_ms2"].abs() <= 8.0) & (df_step2["lat_acc_ms2"].abs() <= 8.0)
df_step3 = df_step2[cond_speed & cond_acc].copy()
n3_pts, n3_veh = len(df_step3), df_step3["source_track_id"].nunique()

# Bước 4: Loại bỏ các quỹ đạo quá ngắn (< 5 giây theo dõi)
track_lengths = df_step3.groupby("source_track_id")["time_sec"].transform("count")
df_clean = df_step3[track_lengths >= 5].copy()
df_clean = df_clean.sort_values(["source_track_id", "time_sec"]).reset_index(drop=True)
n4_pts, n4_veh = len(df_clean), df_clean["source_track_id"].nunique()

# Bước 5: Chuyển đổi tọa độ (lat, lon) -> (x_pos, y_pos) tính bằng mét (m) và tính các cột vật lý chuẩn CSDL
lat_min = float(df_clean["lat"].min())
lon_min = float(df_clean["lon"].min())
lat0_rad = float(np.radians(df_clean["lat"].mean()))

df_clean["x_pos"] = ((df_clean["lon"] - lon_min) * 111320.0 * np.cos(lat0_rad)).round(3)
df_clean["y_pos"] = ((df_clean["lat"] - lat_min) * 110540.0).round(3)
df_clean["speed_ms"] = (df_clean["speed_kmh"] / 3.6).round(2)
df_clean["acc_ms2"] = df_clean["lon_acc_ms2"].round(2)
df_clean["frame_id"] = (df_clean["time_sec"] * 25).round().astype(int)
df_clean["timestamp_ms"] = (df_clean["time_sec"] * 1000).round().astype(int)

# Tính góc hướng di chuyển heading_deg từ độ lệch (dx, dy) giữa 2 thời điểm liên tiếp của từng xe
dx = df_clean.groupby("source_track_id")["x_pos"].diff().fillna(0.0)
dy = df_clean.groupby("source_track_id")["y_pos"].diff().fillna(0.0)
df_clean["heading_deg"] = np.degrees(np.arctan2(dy, dx)).round(2)

# Bảng tổng hợp kết quả làm sạch dữ liệu
cleaning_report = pd.DataFrame([
    {"Bước tiền xử lý": "0. Dữ liệu gốc (sau khi chuyển sang 1 Hz)",              "Số điểm còn lại": n0_pts, "Số điểm bị lọc": 0,              "Số xe (Tracks) còn lại": n0_veh},
    {"Bước tiền xử lý": "1. Bỏ giá trị thiếu & trùng lặp (dropna, drop_duplicates)","Số điểm còn lại": n1_pts, "Số điểm bị lọc": n0_pts - n1_pts, "Số xe (Tracks) còn lại": n1_veh},
    {"Bước tiền xử lý": "2. Lọc bỏ xe đỗ tĩnh ven đường (traveled_d < 5m)",        "Số điểm còn lại": n2_pts, "Số điểm bị lọc": n1_pts - n2_pts, "Số xe (Tracks) còn lại": n2_veh},
    {"Bước tiền xử lý": "3. Lọc ngoại lai vận tốc (>80km/h) & gia tốc (>8m/s²)",   "Số điểm còn lại": n3_pts, "Số điểm bị lọc": n2_pts - n3_pts, "Số xe (Tracks) còn lại": n3_veh},
    {"Bước tiền xử lý": "4. Lọc bỏ quỹ đạo đứt quãng quá ngắn (< 5 giây)",         "Số điểm còn lại": n4_pts, "Số điểm bị lọc": n3_pts - n4_pts, "Số xe (Tracks) còn lại": n4_veh},
])

print("🧹 BẢNG BÁO CÁO KẾT QUẢ TIỀN XỬ LÝ & LÀM SẠCH FILE `20181029_dX_0830_0900.csv`:")
display(cleaning_report)

print("\n✅ Xem 5 dòng đầu của dữ liệu sau khi đã chuẩn hóa tọa độ mét (x_pos, y_pos) và các đại lượng vật lý:")
display(df_clean[["source_track_id", "vehicle_type", "frame_id", "timestamp_ms", "x_pos", "y_pos", "speed_kmh", "speed_ms", "acc_ms2", "heading_deg"]].head(5))

In [ ]:
from matplotlib.patches import Rectangle

# Định nghĩa 6 vùng không gian (zone_name, x_min, x_max, y_min, y_max, chiều dài đường đại diện km)
ZONES_DEF = [
    ("Khu_Trung_Tam", 450.0,  900.0,  650.0, 1400.0, 1.50),
    ("Nhanh_Bac",     300.0, 1000.0, 1400.0, 1900.0, 1.00),
    ("Nhanh_Nam",     300.0, 1000.0,    0.0,  650.0, 1.00),
    ("Nhanh_Tay",       0.0,  450.0,  500.0, 1500.0, 0.90),
    ("Nhanh_Dong",    900.0, 1315.0,  500.0, 1500.0, 0.90),
    ("Toan_Khu_Vuc",    0.0, 1315.0,    0.0, 1900.0, 5.30),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Biểu đồ 1: Cơ cấu phương tiện sau làm sạch
veh_counts = df_clean.groupby("vehicle_type")["source_track_id"].nunique().sort_values(ascending=True)
axes[0, 0].barh(veh_counts.index, veh_counts.values, color="#2a9d8f", edgecolor="black")
for i, v in enumerate(veh_counts.values):
    axes[0, 0].text(v + 40, i, f"{v:,} xe", va="center", fontsize=9)
axes[0, 0].set_xlim(0, veh_counts.max() * 1.18)
axes[0, 0].set_title("1. Số lượng phương tiện theo loại sau làm sạch (Tổng: 8,255 xe)")
axes[0, 0].set_xlabel("Số lượng phương tiện (Tracks)")

# Biểu đồ 2: Phân phối vận tốc (km/h) sau khi lọc nhiễu
axes[0, 1].hist(df_clean["speed_kmh"], bins=35, color="#457b9d", edgecolor="white")
med_spd = df_clean["speed_kmh"].median()
mean_spd = df_clean["speed_kmh"].mean()
axes[0, 1].axvline(med_spd, color="#e63946", linestyle="--", lw=1.8, label=f"Trung vị: {med_spd:.1f} km/h")
axes[0, 1].axvline(mean_spd, color="#ffb703", linestyle="-", lw=1.8, label=f"Trung bình: {mean_spd:.1f} km/h")
axes[0, 1].set_title("2. Phân phối vận tốc phương tiện sau tiền xử lý")
axes[0, 1].set_xlabel("Vận tốc (km/h)")
axes[0, 1].set_ylabel("Số điểm quỹ đạo (1 Hz)")
axes[0, 1].legend()

# Biểu đồ 3: Phân phối gia tốc dọc (lon_acc_ms2) trước và sau khi lọc ngoại lai
axes[1, 0].hist(df_clean["acc_ms2"], bins=40, color="#f4a261", edgecolor="white")
axes[1, 0].set_title("3. Phân phối gia tốc dọc (acc_ms2) nằm gọn trong [-8, +8] m/s²")
axes[1, 0].set_xlabel("Gia tốc (m/s²)")
axes[1, 0].set_ylabel("Số điểm quỹ đạo")

# Biểu đồ 4: Bản đồ không gian quỹ đạo (x_pos, y_pos) và ranh giới 5 vùng phân tích
sample_map = df_clean.iloc[::25]  # Lấy mẫu vẽ điểm tọa độ cho rõ nét
axes[1, 1].scatter(sample_map["x_pos"], sample_map["y_pos"], s=1.0, alpha=0.25, color="#1d3557", label="Điểm quỹ đạo xe")
zone_colors = ["#e63946", "#2a9d8f", "#ffb703", "#8338ec", "#fb8500"]
for (zname, xmin, xmax, ymin, ymax, _), zcol in zip(ZONES_DEF[:5], zone_colors):
    rect = Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor=zcol, lw=2.0, linestyle="--")
    axes[1, 1].add_patch(rect)
    axes[1, 1].text((xmin + xmax) / 2, (ymin + ymax) / 2, zname, color=zcol, fontsize=8, fontweight="bold",
                    ha="center", va="center", bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))
axes[1, 1].set_title("4. Bản đồ quỹ đạo (x_pos, y_pos) & Ranh giới các Vùng (TrafficSpatialZones)")
axes[1, 1].set_xlabel("Tọa độ X - Đông Tây (m)")
axes[1, 1].set_ylabel("Tọa độ Y - Nam Bắc (m)")

plt.tight_layout()
plt.show()

In [ ]:
conn = sqlite3.connect(DB_PATH)

# ------------------------------------------------------------------
# 1. NẠP BẢNG `Tracks` (8,255 phương tiện đã làm sạch)
# ------------------------------------------------------------------
tracks_table = (
    df_clean.groupby("source_track_id", as_index=False)
    .agg(
        vehicle_type=("vehicle_type", "first"),
        total_frames=("frame_id", "count"),
        initial_frame=("frame_id", "min"),
        final_frame=("frame_id", "max"),
    )
)
tracks_table["dataset_id"] = 1
tracks_table = tracks_table[["dataset_id", "source_track_id", "vehicle_type", "total_frames", "initial_frame", "final_frame"]]

conn.execute("DELETE FROM TrajectoryPoints")
conn.execute("DELETE FROM Tracks")
tracks_table.to_sql("Tracks", conn, if_exists="append", index=False)

# Xuất file CSV danh sách phương tiện đã làm sạch
tracks_csv_path = WORK_DIR / "20181029_dX_0830_0900_tracks_cleaned.csv"
tracks_table.to_csv(tracks_csv_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------------
# 2. NẠP BẢNG `TrajectoryPoints` (30,000 điểm đại diện cách đều)
# ------------------------------------------------------------------
db_tracks = pd.read_sql("SELECT track_id, source_track_id FROM Tracks WHERE dataset_id = 1", conn)
pts_merged = df_clean.merge(db_tracks, on="source_track_id", how="inner")

# Lấy mẫu cách đều 30,000 điểm trên toàn bộ các xe để lưu vào SQLite
step_sample = max(1, len(pts_merged) // 30000)
pts_sample = pts_merged.iloc[::step_sample].head(30000).copy()

pts_table = pd.DataFrame({
    "track_id": pts_sample["track_id"],
    "frame_id": pts_sample["frame_id"],
    "timestamp_ms": pts_sample["timestamp_ms"],
    "x_pos": pts_sample["x_pos"],
    "y_pos": pts_sample["y_pos"],
    "speed_ms": pts_sample["speed_ms"],
    "acc_ms2": pts_sample["acc_ms2"],
    "heading_deg": pts_sample["heading_deg"],
})
pts_table.to_sql("TrajectoryPoints", conn, if_exists="append", index=False)

# Xuất file CSV mẫu 30,000 điểm quỹ đạo đã làm sạch
points_csv_path = WORK_DIR / "20181029_dX_0830_0900_points_cleaned_sample.csv"
pts_table.to_csv(points_csv_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------------
# 3. NẠP BẢNG `TrafficSpatialZones` (6 vùng phân tích)
# ------------------------------------------------------------------
conn.execute("DELETE FROM TrafficAggregatedMetrics")
conn.execute("DELETE FROM TrafficSpatialZones")

zones_df = pd.DataFrame([
    {"dataset_id": 1, "zone_name": z[0], "x_min": z[1], "x_max": z[2], "y_min": z[3], "y_max": z[4]}
    for z in ZONES_DEF
])
zones_df.to_sql("TrafficSpatialZones", conn, if_exists="append", index=False)
db_zones = pd.read_sql("SELECT zone_id, zone_name, x_min, x_max, y_min, y_max FROM TrafficSpatialZones", conn)
zone_id_map = dict(zip(db_zones["zone_name"], db_zones["zone_id"]))

# ------------------------------------------------------------------
# 4. TÍNH TOÁN & NẠP BẢNG `TrafficAggregatedMetrics` (10s, 15s, 30s, 60s)
# ------------------------------------------------------------------
def classify_congestion_athens(speed_kmh: float) -> str:
    '''Phân loại 4 mức độ ùn tắc chuẩn trong 01_init_schema.sql theo vận tốc trung bình đô thị'''
    if pd.isna(speed_kmh) or speed_kmh >= 22.0:
        return "Free Flow"
    elif speed_kmh >= 15.0:
        return "Moderate"
    elif speed_kmh >= 10.0:
        return "Heavy"
    else:
        return "Gridlock"

start_dt = pd.Timestamp("2018-10-29 08:30:00")
max_time_s = float(df_clean["time_sec"].max())

agg_rows = []
for interval_s in [10, 15, 30, 60]:
    n_bins = int(max_time_s // interval_s)
    df_clean["bin_idx"] = (df_clean["time_sec"] // interval_s).astype(int)
    df_valid_bins = df_clean[df_clean["bin_idx"] < n_bins]

    for zname, xmin, xmax, ymin, ymax, road_len_km in ZONES_DEF:
        zid = zone_id_map[zname]
        sub_z = df_valid_bins[
            (df_valid_bins["x_pos"] >= xmin) & (df_valid_bins["x_pos"] <= xmax) &
            (df_valid_bins["y_pos"] >= ymin) & (df_valid_bins["y_pos"] <= ymax)
        ]
        grp = sub_z.groupby("bin_idx").agg(
            vehicle_count=("source_track_id", "nunique"),
            total_pts_1hz=("source_track_id", "count"),
            avg_speed_kmh=("speed_kmh", "mean"),
        ).reindex(range(n_bins), fill_value=0)

        # Số xe trung bình hiện diện trong 1 giây = total_pts_1hz / interval_s
        # Mật độ giao thông (xe/km) = Số xe trung bình hiện diện / Chiều dài mạng lưới đường đại diện (km)
        density_series = ((grp["total_pts_1hz"] / float(interval_s)) / road_len_km).round(2)
        speed_series = grp["avg_speed_kmh"].round(2)

        # Đánh dấu khung giờ cao điểm (top 20% mật độ cao nhất của vùng)
        peak_threshold = float(density_series.quantile(0.80))
        is_peak_series = ((density_series >= peak_threshold) & (density_series > 0)).astype(int)

        for b_idx in range(n_bins):
            w_start = start_dt + pd.Timedelta(seconds=b_idx * interval_s)
            w_end = w_start + pd.Timedelta(seconds=interval_s)
            v_cnt = int(grp.loc[b_idx, "vehicle_count"])
            v_spd = float(speed_series.loc[b_idx]) if v_cnt > 0 else 0.0
            v_den = float(density_series.loc[b_idx])
            v_peak = int(is_peak_series.loc[b_idx])
            v_cong = classify_congestion_athens(v_spd) if v_cnt > 0 else "Free Flow"

            agg_rows.append({
                "dataset_id": 1,
                "zone_id": zid,
                "window_interval_sec": interval_s,
                "window_start_time": w_start.strftime("%Y-%m-%d %H:%M:%S"),
                "window_end_time": w_end.strftime("%Y-%m-%d %H:%M:%S"),
                "vehicle_count": v_cnt,
                "avg_speed_kmh": v_spd,
                "traffic_density_veh_km": v_den,
                "is_peak_period": v_peak,
                "congestion_level": v_cong,
            })

agg_df = pd.DataFrame(agg_rows)
agg_df.to_sql("TrafficAggregatedMetrics", conn, if_exists="append", index=False)
conn.commit()

# Xuất file CSV chuỗi thời gian tổng hợp
agg_csv_path = WORK_DIR / "20181029_dX_0830_0900_aggregated_metrics.csv"
agg_df.to_csv(agg_csv_path, index=False, encoding="utf-8-sig")

print("✅ HOÀN TẤT NẠP DỮ LIỆU TỪ `20181029_dX_0830_0900.csv` VÀO CSDL & XUẤT FILE CSV SẠCH:")
print(f"   1. Bảng `Tracks`                   : {len(tracks_table):,} phương tiện  -> Đã lưu CSV: {tracks_csv_path.name}")
print(f"   2. Bảng `TrajectoryPoints`         : {len(pts_table):,} điểm tọa độ -> Đã lưu CSV: {points_csv_path.name}")
print(f"   3. Bảng `TrafficSpatialZones`      : {len(db_zones):,} vùng không gian")
print(f"   4. Bảng `TrafficAggregatedMetrics` : {len(agg_df):,} bản ghi chuỗi thời gian -> Đã lưu CSV: {agg_csv_path.name}")

print("\n🔍 Kiểm tra 5 dòng đầu của bảng `Tracks` trong CSDL:")
display(pd.read_sql("SELECT * FROM Tracks LIMIT 5", conn))

print("\n🔍 Kiểm tra 5 dòng đầu của bảng `TrajectoryPoints` trong CSDL:")
display(pd.read_sql("SELECT * FROM TrajectoryPoints LIMIT 5", conn))

print("\n🔍 Kiểm tra bảng `TrafficSpatialZones` trong CSDL:")
display(db_zones)

print("\n🔍 Kiểm tra 6 dòng đầu của bảng `TrafficAggregatedMetrics` (cửa sổ 30s) trong CSDL:")
display(pd.read_sql('''
    SELECT m.metric_id, z.zone_name, m.window_interval_sec, m.window_start_time,
           m.vehicle_count, m.avg_speed_kmh, m.traffic_density_veh_km, m.is_peak_period, m.congestion_level
    FROM TrafficAggregatedMetrics m
    JOIN TrafficSpatialZones z ON m.zone_id = z.zone_id
    WHERE m.window_interval_sec = 30
    LIMIT 6
''', conn))
conn.close()